In [1]:
from pypdf import PdfReader

pdf_path="data/raw_data.pdf"

reader=PdfReader(pdf_path)
all_pages=""    
for page in reader.pages:
    all_pages+=page.extract_text()


In [2]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter  

def load_pdf_files(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    
    # text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    # split_documents = text_splitter.split_documents(documents)
    documents = loader.load()   
    return documents
extracted_docs=load_pdf_files("data")


c:\My Project\project1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from typing import List
from langchain.schema import Document

def filter_to_minmal_doc(documents: List[Document]) -> List[Document]:
    minimal_docs : List[Document]= []
    for doc in documents:
        src=doc.metadata.get("source")
        minimal_docs.append(Document(page_content=doc.page_content,metadata={"source": src}))
    return minimal_docs

minimal_doc=filter_to_minmal_doc(extracted_docs)

In [4]:
def split_extraced_docs(minimal_doc):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(minimal_doc)
    return text_chunks

text_chunks=split_extraced_docs(minimal_doc) 

In [5]:
from langchain.embeddings import HuggingFaceEmbeddings

def create_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
   # text_embeddings = embeddings.embed_documents([chunk.page_content for chunk in text_chunks])
    return embeddings


embedding=create_embeddings()

C:\Users\ankchauhan\AppData\Local\Temp\ipykernel_79688\3936673434.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [6]:
from dotenv import load_dotenv
import os
load_dotenv()
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY") 
#PINECONE_INDEX_NAME=os.getenv("PINECONE_INDEX_NAME")
os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY




In [7]:
# import os
# key = os.getenv("OPENAI_API_KEY")
# print("is None?", key is None)
# print("repr:", repr(key))
# print("length:", 0 if key is None else len(key))

In [8]:
from pinecone import Pinecone
pinecone_api_key=PINECONE_API_KEY

pc=Pinecone(apikey=pinecone_api_key)
pc

In [9]:
from pinecone import ServerlessSpec

index_name="medical-chatboat"

if not pc.has_index(index_name):
    pc.create_index(name=index_name, 
                    dimension=384, 
                    metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))     


index=pc.Index(index_name)


In [10]:
from pinecone import Pinecone
from langchain_pinecone import  PineconeVectorStore 
text_chunks=text_chunks[:10]

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks, 
    embedding=embedding,
    index_name=index_name)

In [11]:
from langchain_pinecone import  PineconeVectorStore 
#embed each chunk into vector db
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name, 
    embedding=embedding)
    



In [12]:
retriever=docsearch.as_retriever(search_kwargs={"k":3}, search_type="similarity")
retriever   

VectorStoreRetriever(tags=['PineconeVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000002533E2A2E90>, search_kwargs={'k': 3})

In [13]:
retrieved_docs=docsearch.similarity_search("how to cure pimples?", k=3)


In [25]:
# from langchain_openai import ChatOpenAI
# chatmodel=ChatOpenAI(model="gpt-4o", temperature=0.9)

from langchain_community.llms import Ollama

llm = Ollama(
    model="llama3.2:3b",
    base_url="http://localhost:11434"
)

# from langchain_community.chat_models import ChatOllama
# chatmodel = ChatOllama(model="llama3.1", temperature=0.9)

In [26]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate



In [27]:
system_prompt=(
    "You are an medical assistant. You will be provided with some medical documents."
    " Please answer the question based on the provided documents. "
    "If you don't know the answer, say you don't know."
    "use the following pieces of information to answer the question:"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
        [
        ("system", system_prompt),
        ("human", "{input}")
        ]
    )

In [28]:
question_answer_chain=create_stuff_documents_chain(llm, prompt)
rag_chain=create_retrieval_chain(retriever, question_answer_chain)
# print(OPENAI_API_KEY)



In [29]:


print(llm("Hello, can you summarize what acne is?"))

Acne is a common skin condition that affects millions of people worldwide. Here's a brief summary:

**What is acne?**

Acne is a chronic inflammatory skin disease characterized by the occurrence of comedones (blackheads and whiteheads), papules, pustules, nodules, and cysts on the skin.

**Causes:**

1. **Clogged pores**: Dead skin cells, oil, and bacteria can clog pores, leading to inflammation.
2. **Hormonal fluctuations**: Hormonal changes during puberty, menstruation, pregnancy, and menopause can cause acne.
3. **Bacterial overgrowth**: The bacterium Propionibacterium acnes (P. acnes) contributes to acne development.

**Symptoms:**

1. Blackheads (comedo necroticus)
2. Whiteheads (comedo congenital)
3. Papules
4. Pustules
5. Nodules
6. Cysts

**Types of acne:**

1. **Non-inflammatory**: Mild, superficial acne.
2. **Inflammatory**: More severe forms, including nodules and cysts.

Acne can be treated with various treatments, such as topical creams, oral medications, and lifestyle cha

In [ ]:
response = rag_chain.invoke({"input": "could you please tell me about fever?"})
print(response["answer"])  # This should print the answer directly

As a medical assistant, I'd be happy to help you with your question.

Allergies occur when the body's immune system overreacts to a harmless substance, known as an allergen. When an allergen enters the body, it triggers the release of histamine and other chemicals, leading to symptoms such as itching, swelling, congestion, and sneezing.

There are several common types of allergies, including:

1. Allergic rhinitis: This is a condition where the nasal membranes become inflamed and swollen due to sensitivity to airborne matter like pollen or cat hair.
2. Insect sting allergy: Some people may develop an allergic reaction to insect stings, which can cause symptoms such as hives, itching, and swelling.

If you're experiencing symptoms of allergies, there are several treatment options available. One common approach is to use saline solutions to help alleviate symptoms. Saline nasal sprays or drops can help moisturize the nasal membranes and reduce congestion.

It's also important to identify